# 参考資料：投稿画像のダウンロード

> **この Notebook は授業の本編ではありません（参考資料）．**
> SNS の画像には **センシティブな内容（性的・暴力的・不快な画像）** が含まれることが分かっています．
> 画像を扱うプロジェクトを行う場合は，事前に教員に相談し，次の点を守ってください．
>
> - モデレーション／自己申告のラベル（`labels` 列）が付いた投稿は除外する（`download_images()` の既定動作）．ただしラベルが無くてもセンシティブな画像はあり得る．
> - まず少量（10〜20枚）を取得して目視で確認し，問題があれば話題（検索語）を変える．公式アカウント（自治体・報道機関）の投稿に限定するのも有効．
> - 取得した画像は授業内の分析にだけ使い，再配布しない．発表には画像そのものではなく集計結果を載せる．
> - 不快な画像を見てしまった場合は無理に続けず，教員に伝える．

## 準備

第2回と同じく `bsky_config.ini` を設定したうえで，`sns_analysis` フォルダから相対パスで実行します（この Notebook は `sns_analysis/reference/` にあるので，最初のセルで作業フォルダを1つ上に移します）．

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "reference":     # sns_analysis/ を作業フォルダにする
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
import matplotlib.pyplot as plt
from bsky_utils import *

pd.set_option("display.max_colwidth", 60)
set_japanese_font()
login_status()

### 投稿データの読み込み

第2回で保存した `data/posts.csv` を使います（無ければ `search_posts_paged` で集めてください）．

In [ ]:
df = load_posts("data/posts.csv")
print("ラベル付き投稿:", int((df["labels"].fillna("") != "").sum()) if "labels" in df else "（labels 列なし：第2回 Notebook を新しい bsky_utils で再実行してください）")
df[["query", "text", "image_count", "labels"]].head() if "labels" in df else df[["query", "text", "image_count"]].head()

## 画像について取得できるもの

| 項目 | 列名 | 説明 |
|---|---|---|
| 枚数 | `image_count` | 投稿に付いた画像の数（0〜4） |
| 画像URL（縮小版） | `image_thumbs` | 長辺 1000px 程度・数十KB．分析にはこれで十分 |
| 画像URL（原寸） | `image_urls` | 最大 2000px・数百KB．細部を見たいとき |
| 代替テキスト | `image_alts` | 投稿者が付けた画像の説明文（付いていないことも多い） |

複数枚のときは `|` 区切りで入っています．画像はサーバから webp 形式で届くので，扱いやすい JPEG に変換して保存します．

### 守ってほしいこと

- 画像には投稿者の著作権があります．**授業内の分析にだけ使い，再配布しない**（`data/` は Git 管理外）．
- 発表資料には画像そのものを並べず，**集計結果**（検出した物体の種類と数，明るさの分布など）で示す．必要な場合も枚数を絞り，人物が写るものは避ける．
- 一度に大量に取得しない（関数の中で1枚ごとに待ち時間を入れています）．

## 1. 画像付き投稿がどれくらいあるか

まず，読み込んだデータ（`df`）で画像付き投稿の割合と枚数の分布を確認します．話題によって大きく違います．

In [ ]:
img_posts = df[df["image_count"] > 0]
print("画像付き投稿:", len(img_posts), "件 /", len(df), "件 （", round(len(img_posts) / len(df) * 100, 1), "%）")
print("画像の総数:", int(df["image_count"].sum()), "枚")

summary = df.groupby("query")["image_count"].agg(投稿数="size", 画像付き=lambda s: int((s > 0).sum()), 画像枚数="sum")
summary["画像付き割合(%)"] = (summary["画像付き"] / summary["投稿数"] * 100).round(1)
summary

In [ ]:
# 1投稿あたりの枚数の分布
df["image_count"].value_counts().sort_index().rename_axis("枚数").rename("投稿数")

## 2. 画像が付きやすい話題で集める

写真が多い話題（「桜」「ランチ」「旅行」「猫」など）で集めると，画像分析の練習がしやすくなります．

In [ ]:
df_photo = search_posts_paged("写真", max_posts=200)
print("画像付き投稿:", int((df_photo["image_count"] > 0).sum()), "件 / 画像の総数:", int(df_photo["image_count"].sum()), "枚")
df_photo[df_photo["image_count"] > 0][["created_at_jst", "text", "image_count", "like_count"]].head()

## 3. ダウンロードする

`download_images(表, out_dir=保存先, max_posts=投稿数の上限, size="thumb"|"fullsize", all_images=True|False)`

- 授業では `max_posts=20` 程度に絞ります．プロジェクトで増やすときは `scripts/download_images.py` を使います．
- 結果は **1枚1行** の表で，`out_dir/images.csv` にも保存されます（第4回の物体検出で使います）．
- 保存済みの画像は再取得しないので，同じセルを何度実行しても構いません．

In [ ]:
images = download_images(df_photo, out_dir="data/images", max_posts=20, size="thumb")
images.head(8)

## 4. 一覧表示する

`show_images(表, n=枚数, cols=列数, caption=列名)` で格子状に表示します．`caption="alt"` にすると代替テキストが，`caption="post_id"` にすると投稿IDが添えられます．

In [ ]:
show_images(images, n=12, cols=4, caption="alt")

## 5. 代替テキスト（alt）を見る

代替テキストは投稿者が画像に付けた説明文です（視覚障害のある利用者のための機能）．付いている投稿は多くありませんが，**画像の内容を文字で表したデータ**として，テキスト分析（第3回）と組み合わせられます．

In [ ]:
with_alt = images[images["alt"].fillna("") != ""]
print("代替テキスト付き:", len(with_alt), "枚 /", len(images), "枚")
with_alt[["post_id", "image_index", "alt"]].head(10)

## 6. 縮小版（thumb）と原寸（fullsize）の違い

1投稿だけ原寸でも取得して，大きさと容量を比べます．分析の多く（物体検出，明るさ・色の集計）は縮小版で十分です．原寸は容量が10倍近くになるので，必要なときだけ使います．

In [ ]:
one = df_photo[df_photo["image_count"] > 0].head(1)
full = download_images(one, out_dir="data/images_fullsize", max_posts=1, size="fullsize", verbose=False)
thumb = images[images["post_id"] == one.iloc[0]["post_id"]]

pd.concat([thumb.assign(種類="thumb"), full.assign(種類="fullsize")])[["種類", "image_index", "width", "height", "bytes"]]

## 7. 保存先の構造と，まとめてダウンロードするスクリプト

```
data/images/
├── images.csv                 ← 1枚1行の一覧（post_id, image_index, image_path, alt, width, height, bytes, size）
├── 3f2a9c1d7e5b_1.jpg         ← {post_id}_{何枚目}.jpg
├── 3f2a9c1d7e5b_2.jpg
└── ...
```

プロジェクトで数百枚を集めるときは，Anaconda Prompt からスクリプトで実行します（保存済みは飛ばすので途中で止めても再開できます）．

```bash
python scripts/download_images.py --csv data/posts.csv --out data/images --max-posts 100
python scripts/download_images.py --csv data/posts.csv --query 観光 --size fullsize --max-posts 50
python scripts/download_images.py --csv data/posts.csv --first-only --max-images 200
```

`images.csv` の `post_id` で投稿の表（本文・反応数）と結合すれば，「画像付き投稿の反応」「画像の内容と本文」のような分析ができます（第4回）．

In [ ]:
# 画像の表と投稿の表を post_id で結合する例
merged = images.merge(df_photo[["post_id", "text", "like_count", "repost_count"]], on="post_id", how="left")
merged[["post_id", "image_index", "alt", "like_count", "text"]].head()

## ラベル付き投稿の扱い

`labels` 列には，Bluesky のモデレーションや投稿者自身が付けたラベル（例：`porn`，`sexual`，`nudity`，`graphic-media`）が入ります．
`download_images()` は既定でこれらを除外します（`skip_labeled=False` で含められますが，授業では使いません）．
ラベルの有無だけでは安全は保証されないため，**取得したら必ず少量を目視で確認**してください．

In [ ]:
if "labels" in df:
    print(df["labels"].fillna("").replace("", "（なし）").value_counts().head(10))